# Multi-Agent AI Research Studio

## 1. Introduction

Multi-Agent AI Research Studio is a local application where multiple specialized AI agents work together to research a topic, analyze evidence, and produce usable business outputs.

**Business value:** it helps teams move from a broad question to cited answers, reports, checklists, file summaries, and image interpretations faster than a manual research workflow.

**Why multi-agent systems matter:** different roles can focus on different parts of the work. A researcher gathers evidence, an analyst evaluates it, and a writer turns it into a polished deliverable.

## 2. Technology Overview

- **Streamlit** provides the browser-based interface.
- **CrewAI** coordinates agents, tasks, and workflows.
- **Google Gemini** powers text generation and image understanding.
- **Tavily** provides web search results and source URLs for research.

## 3. Environment Setup

Install dependencies with:

```bash
pip install -r requirements.txt
```

Create a `.env` file from `.env.example` and add keys:

```env
GOOGLE_API_KEY=your_google_api_key
TAVILY_API_KEY=your_tavily_api_key
```

API keys should stay in `.env`, not inside notebooks, screenshots, or source code.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
google_key_loaded = bool(os.getenv("GOOGLE_API_KEY"))
tavily_key_loaded = bool(os.getenv("TAVILY_API_KEY"))
print({"GOOGLE_API_KEY loaded": google_key_loaded, "TAVILY_API_KEY loaded": tavily_key_loaded})

## 4. Gemini Demonstration

The production app initializes Gemini once and reuses it. This improves speed and avoids repeated setup work.

In [ ]:
import os
from crewai import LLM

llm = LLM(
    model=os.getenv("GEMINI_TEXT_MODEL", "gemini/gemini-2.5-flash"),
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.25,
)
print("Reusable Gemini LLM object created")

## 5. Tavily Demonstration

Tavily brings current web evidence into the research workflow. The app caches repeated searches so the same topic does not trigger unnecessary duplicate calls.

In [ ]:
import os
from tavily import TavilyClient

if os.getenv("TAVILY_API_KEY"):
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    results = client.search("benefits of multi-agent AI systems", max_results=3).get("results", [])
    for item in results:
        print(item.get("title"), "-", item.get("url"))
else:
    print("Add TAVILY_API_KEY to .env to run this example.")

## 6. CrewAI Fundamentals

- **Agents** are role-based AI workers.
- **Tasks** describe work assigned to agents.
- **Crews** coordinate agents and tasks into a workflow.

The example below creates one agent and one task. It is intentionally small so the concept is easy to inspect.

In [ ]:
from crewai import Agent, Crew, Process, Task

if os.getenv("GOOGLE_API_KEY"):
    explainer = Agent(
        role="Concept Explainer",
        goal="Explain business concepts clearly",
        backstory="You explain technical ideas to non-technical stakeholders.",
        llm=llm,
        verbose=False,
    )
    task = Task(
        description="Explain multi-agent AI in three bullet points.",
        expected_output="Three clear bullet points.",
        agent=explainer,
    )
    print(Crew(agents=[explainer], tasks=[task], process=Process.sequential, verbose=False).kickoff())
else:
    print("Add GOOGLE_API_KEY to .env to run this example.")

## 7. Mode Demonstrations

These simplified examples mirror the application modes without duplicating the entire Streamlit codebase.

### Quick Answer

One agent answers a question using a focused prompt and web evidence.

In [ ]:
quick_answer_prompt = "Answer briefly: What are AI agents useful for in customer support?"
print(quick_answer_prompt)

### Full Report

A sequential workflow passes work from research to analysis to writing.

In [ ]:
full_report_sections = ["Executive Summary", "Key Findings", "Analysis", "Recommendations", "Sources"]
print("Full report sections:", full_report_sections)

### Two Outputs at Once

The app runs a glossary creator and checklist creator simultaneously, then displays them side by side.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def demo_glossary(topic):
    return f"Glossary for {topic}: agent, task, crew"

def demo_checklist(topic):
    return f"Checklist for {topic}: define goal, gather evidence, review output"

with ThreadPoolExecutor(max_workers=2) as executor:
    a = executor.submit(demo_glossary, "AI research")
    b = executor.submit(demo_checklist, "AI research")
    print(a.result())
    print(b.result())

### File Analysis

The app previews CSV files and limits TXT content before sending it to an analysis agent. This keeps the workflow fast and laptop-friendly.

In [ ]:
import pandas as pd

sample = pd.DataFrame({"channel": ["web", "email", "events"], "leads": [120, 80, 45]})
print(sample.head().to_markdown(index=False))

### Deep Research

Deep research uses three roles: Senior Researcher, Strategic Analyst, and Research Author. It emphasizes multiple viewpoints, uncertainty, and confidence levels.

In [ ]:
confidence_levels = ["High", "Medium", "Low"]
print("Allowed confidence ratings:", confidence_levels)

### Image Analysis

Image analysis uses Gemini Vision directly. Before analysis, large images are resized to reduce token usage while preserving readability.

In [ ]:
from PIL import Image

def resize_for_vision(image, max_side=1400):
    image = image.convert("RGB")
    image.thumbnail((max_side, max_side))
    return image

print("Image helper ready")

## 8. Workflow Visualization

### Sequential Workflow

```mermaid
flowchart LR
    A[Research Agent] --> B[Analysis Agent]
    B --> C[Writer Agent]
```

### Parallel Workflow

```mermaid
flowchart LR
    T[User Topic] --> G[Glossary Agent]
    T --> C[Checklist Agent]
```

### Deep Research Workflow

```mermaid
flowchart LR
    S[Senior Researcher] --> A[Strategic Analyst]
    A --> R[Research Author]
```

## 9. Best Practices

- Reuse shared resources such as Gemini and Tavily clients.
- Use a central agent factory to keep roles consistent.
- Cache repeated searches to reduce latency and API usage.
- Validate user inputs before running expensive workflows.
- Show friendly errors to users and log details for maintainers.
- Keep prompts structured so generated outputs are predictable.
- Scale workflows only when the task benefits from additional agents.

## 10. Conclusion

Multi-Agent AI Research Studio demonstrates how AI collaboration can turn scattered questions into structured outputs. The business impact is faster research, more consistent reporting, and reusable workflows that can be extended to new teams, domains, and data sources.